# 📊 Bước 2: Descriptive Analytics
Phân tích mô tả dữ liệu mua sắm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
CHARTS = '../output/charts'
os.makedirs(CHARTS, exist_ok=True)

# Load dữ liệu đã làm sạch
try:
    df = pd.read_pickle('../output/df_clean.pkl')
except:
    df = pd.read_csv('../data/shopping_trends.csv')
    bins = [0,18,25,35,45,55,100]
    labels = ['<18','18-25','26-35','36-45','46-55','56+']
    df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=labels)
    for col in ['Gender','Category','Season','Subscription Status','Location','Item Purchased']:
        df[col] = df[col].astype('category')

print(f"Dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột")
df.head(3)


## 2.1 KPI Tổng hợp

In [ ]:
print(f"{'Tổng doanh thu':30s}: ${df['Purchase Amount (USD)'].sum():>12,.0f}")
print(f"{'Tổng khách hàng':30s}: {df['Customer ID'].nunique():>12,}")
print(f"{'Giá trị đơn hàng TB':30s}: ${df['Purchase Amount (USD)'].mean():>12.2f}")
print(f"{'Giá trị đơn hàng Median':30s}: ${df['Purchase Amount (USD)'].median():>12.0f}")
print(f"{'Tỷ lệ Subscriber':30s}: {(df['Subscription Status']=='Yes').mean()*100:>11.1f}%")
print(f"{'Điểm đánh giá TB':30s}: {df['Review Rating'].mean():>12.2f}")


## 2.2 Doanh thu theo Giới tính

In [ ]:
gender_rev = df.groupby('Gender', observed=True)['Purchase Amount (USD)'].agg(
    Total='sum', Average='mean', Count='count'
).reset_index().round(2)
display(gender_rev)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(gender_rev['Total'], labels=gender_rev['Gender'].astype(str),
            autopct='%1.1f%%', colors=sns.color_palette('Set2', 2))
axes[0].set_title('Tổng Doanh Thu theo Giới Tính')
axes[1].bar(gender_rev['Gender'].astype(str), gender_rev['Average'],
            color=sns.color_palette('Set2', 2))
axes[1].set_title('Doanh Thu Trung Bình theo Giới Tính')
axes[1].set_ylabel('USD')
plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_01_gender.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.3 Doanh thu theo Nhóm tuổi

In [ ]:
age_rev = df.groupby('Age Group', observed=True)['Purchase Amount (USD)'].agg(
    Total='sum', Average='mean', Count='count'
).reset_index().round(2)
display(age_rev)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(age_rev['Age Group'].astype(str), age_rev['Total'],
       color=sns.color_palette('Set2', len(age_rev)))
ax.set_title('Tổng Doanh Thu theo Nhóm Tuổi', fontsize=13, fontweight='bold')
ax.set_ylabel('USD')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_02_age_group.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.4 Doanh thu theo Danh mục & Mùa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Danh mục
cat_rev = df.groupby('Category', observed=True)['Purchase Amount (USD)'].sum().sort_values(ascending=False)
axes[0].barh(cat_rev.index.astype(str)[::-1], cat_rev.values[::-1],
             color=sns.color_palette('Set2', len(cat_rev)))
axes[0].set_title('Doanh Thu theo Danh Mục')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Mùa
season_rev = df.groupby('Season', observed=True)['Purchase Amount (USD)'].sum().sort_values(ascending=False)
axes[1].bar(season_rev.index.astype(str), season_rev.values,
            color=sns.color_palette('pastel', len(season_rev)))
axes[1].set_title('Doanh Thu theo Mùa')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_03_cat_season.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.5 Top 10 sản phẩm & địa điểm

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top_items = df.groupby('Item Purchased', observed=True)['Purchase Amount (USD)'].sum().nlargest(10)
axes[0].barh(top_items.index.astype(str)[::-1], top_items.values[::-1],
             color=sns.color_palette('Spectral_r', 10))
axes[0].set_title('Top 10 Sản Phẩm Bán Chạy')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

top_locs = df.groupby('Location', observed=True)['Purchase Amount (USD)'].sum().nlargest(10)
axes[1].barh(top_locs.index.astype(str)[::-1], top_locs.values[::-1],
             color=sns.color_palette('Blues_d', 10))
axes[1].set_title('Top 10 Địa Điểm Doanh Thu Cao Nhất')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_04_top10.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.6 Histogram & Boxplot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['Purchase Amount (USD)'], bins=30,
             color='steelblue', edgecolor='white')
axes[0].set_title('Phân Phối Purchase Amount')
axes[0].set_xlabel('USD')

df_plot = df.copy()
df_plot['Category'] = df_plot['Category'].astype(str)
sns.boxplot(data=df_plot, x='Category', y='Purchase Amount (USD)',
            palette='Set2', ax=axes[1])
axes[1].set_title('Boxplot theo Danh Mục')

plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_05_hist_box.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.7 Heatmap Danh mục × Mùa

In [ ]:
pivot = df.pivot_table(values='Purchase Amount (USD)',
                       index='Category', columns='Season',
                       aggfunc='sum', observed=True)
pivot.index   = pivot.index.astype(str)
pivot.columns = pivot.columns.astype(str)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax)
ax.set_title('Heatmap: Doanh Thu theo Danh Mục × Mùa', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_06_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
